**There are 3 types of ways to create a Delta Table.**

**1.First way to create a Delta Table using SQL.**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_sql (
    product_id INT,
    product_name STRING,
    price INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_products_sql VALUES
(1, 'Laptop', 50000),
(2, 'Mouse', 800),
(3, 'Keyboard', 1500)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM retail_products_sql;

product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


**2.Second way to create a Delta Table using Dataframe.**

In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]
 
df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)
 
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")
 
display(spark.table("retail_products_pyspark"))

product_id,product_name,price
4,Monitor,12000
5,Printer,18000
6,Webcam,2500


**3.Third way to create a Delta Table using Query.**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_products_ctas;

product_id,product_name,price
7,Tablet,25000
8,Speaker,5000
9,Router,3500


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);

In [0]:
%sql
MERGE INTO retail_products_sql AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
SELECT * FROM retail_products_sql
ORDER BY product_id;

product_id,product_name,price
2,Wireless Mouse,950
10,USB Hub,1200


In [0]:
%sql 
DESCRIBE DETAIL retail_products_sql;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ba7226cc-fe0b-4555-a7a9-8293f524f6ba,sara_01.default.retail_products_sql,null,abfss://unity-catalog-storage@dbstoragexbnanwpqvp2ji.dfs.core.windows.net/7405608280471537/__unitystorage/catalogs/57e45411-ed71-4d21-b5f6-6072d1cfa318/tables/aa2454ff-efb6-4646-904b-ec1701a99608,2026-11-05T07:14:24.568Z,2026-11-05T08:48:33.000Z,List(),List(),2,2568,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> aa2454ff-efb6-4646-904b-ec1701a99608, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4bc3175f-8318-418a-bf82-49d0dec909a4, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-afbc073e-617e-4659-bb36-e46556c5a394)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
DESCRIBE HISTORY retail_products_sql;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-05T08:48:33.000Z,145261127613895,azuser7416_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(product_id#10147 = product_id#10138)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(935456848385200),9a19bcf6-d2b5-44bf-8737-2ba6656c822d,1105-084803-7q9i5fhw-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2568, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 6483, materializeSourceTimeMs -> 1011, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2323, numTargetRowsUpdated -> 0, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3005)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-11-05T07:14:24.000Z,145261127613895,azuser7416_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""aa2454ff-efb6-4646-904b-ec1701a99608"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4bc3175f-8318-418a-bf82-49d0dec909a4"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-afbc073e-617e-4659-bb36-e46556c5a394""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(935456848385200),a26deec9-a12d-4b7b-93b0-e4b0c2913e6a,1105-070953-dvchz43p-v2n,1,WriteSerializable,false,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-11-05T07:11:20.000Z,145261127613895,azuser7416_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(935456848385200),bc304679-616e-4003-aeb0-661ac8b18978,1105-070953-dvchz43p-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1317)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-11-05T07:10:13.000Z,145261127613895,azuser7416_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""aa2454ff-efb6-4646-904b-ec1701a99608"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4bc3175f-8318-418a-bf82-49d0dec909a4"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-afbc073e-617e-4659-bb36-e46556c5a394""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(9354568

In [0]:
%sql
SELECT *
FROM retail_products_sql VERSION AS OF 1;

product_id,product_name,price
1,Laptop,50000
2,Mouse,800
3,Keyboard,1500


In [0]:
%sql
OPTIMIZE retail_products_sql;

path,metrics
abfss://unity-catalog-storage@dbstoragexbnanwpqvp2ji.dfs.core.windows.net/7405608280471537/__unitystorage/catalogs/57e45411-ed71-4d21-b5f6-6072d1cfa318/tables/aa2454ff-efb6-4646-904b-ec1701a99608,"List(1, 2, List(2107, 2107, 2107.0, 1, 2107), List(1266, 1302, 1284.0, 2, 2568), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1793869616468, 1793869619558, 8, 1, null, List(0, 0), null, 3, 3, 481, 0, null, null, 0)"


In [0]:
%sql
OPTIMIZE retail_products_sql
ZORDER BY (product_id);

path,metrics
abfss://unity-catalog-storage@dbstoragexbnanwpqvp2ji.dfs.core.windows.net/7405608280471537/__unitystorage/catalogs/57e45411-ed71-4d21-b5f6-6072d1cfa318/tables/aa2454ff-efb6-4646-904b-ec1701a99608,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2107), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793869917987, 1793869918752, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"
